In [1]:
! ls drive/MyDrive/CSCI\ 5527\ Project/Trained\ Models/sleep_deprivation_classification/Chinmay

residual_eeg_cnn_model.pt  residual_eeg_cnn_state_dict.pt


In [2]:
import copy
import json
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
class SEBlock(nn.Module):
    # Squeeze excitation learns how important each feature channel is
    # It lets the network emphasize useful channels and reduce weaker ones
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        pooled = self.pool(x).view(b, c)
        scale = self.fc(pooled).view(b, c, 1, 1)
        return x * scale



class ResidualBlock(nn.Module):
    # Residual blocks help deeper networks train by adding a skip connection
    # The block learns a correction to the input instead of learning everything from scratch
    def __init__(self, channels, dropout=0.10):
        super().__init__()
        self.block = nn.Sequential(
            ConvBNAct(channels, channels, kernel_size=3, padding=1, act="silu"),
            nn.Dropout2d(dropout),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        return self.act(x + self.block(x))

class ConvBNAct(nn.Module):
    # This is a small reusable block made of convolution, batch norm and activation
    # Many CNNs in the notebook are built from this block
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, act="relu"):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)

        if act == "relu":
            self.act = nn.ReLU(inplace=True)
        elif act == "leaky_relu":
            self.act = nn.LeakyReLU(0.1, inplace=True)
        elif act == "silu":
            self.act = nn.SiLU(inplace=True)
        else:
            raise ValueError(f"Unsupported activation: {act}")

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

def initialize_custom_model(module):
    for m in module.modules():
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, (nn.BatchNorm2d, nn.LayerNorm)):
            if getattr(m, "weight", None) is not None:
                nn.init.ones_(m.weight)
            if getattr(m, "bias", None) is not None:
                nn.init.zeros_(m.bias)

class ResidualEEGCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.stem = nn.Sequential(
            ConvBNAct(1, 32, kernel_size=(7, 31), padding=(3, 15), act="leaky_relu"),
            nn.MaxPool2d(kernel_size=(2, 4)),
            ConvBNAct(32, 64, kernel_size=(5, 15), padding=(2, 7), act="silu"),
            nn.MaxPool2d(kernel_size=(2, 4)),
        )
        self.res_stack = nn.Sequential(
            ResidualBlock(64, dropout=0.10),
            SEBlock(64),
            ConvBNAct(64, 128, kernel_size=3, padding=1, act="silu"),
            nn.MaxPool2d(kernel_size=(2, 2)),
            ResidualBlock(128, dropout=0.15),
            SEBlock(128),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.LayerNorm(256),
            nn.SiLU(inplace=True),
            nn.Dropout(0.55),
            nn.Linear(256, num_classes),
        )
        initialize_custom_model(self)

    def forward(self, x):
        x = self.stem(x)
        x = self.res_stack(x)
        return self.head(x)


In [4]:
model = ResidualEEGCNN(num_classes=2)

In [5]:
import torch

path = "/content/drive/MyDrive/CSCI 5527 Project/Trained Models/sleep_deprivation_classification/Chinmay/residual_eeg_cnn_state_dict.pt"

state_dict = torch.load(path, map_location="cpu", weights_only=False)

model = ResidualEEGCNN(num_classes=2)
model.load_state_dict(state_dict)
model.eval()

print("Model loaded successfully.")

Model loaded successfully.


In [6]:
model

ResidualEEGCNN(
  (stem): Sequential(
    (0): ConvBNAct(
      (conv): Conv2d(1, 32, kernel_size=(7, 31), stride=(1, 1), padding=(3, 15), bias=False)
      (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act): LeakyReLU(negative_slope=0.1, inplace=True)
    )
    (1): MaxPool2d(kernel_size=(2, 4), stride=(2, 4), padding=0, dilation=1, ceil_mode=False)
    (2): ConvBNAct(
      (conv): Conv2d(32, 64, kernel_size=(5, 15), stride=(1, 1), padding=(2, 7), bias=False)
      (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (3): MaxPool2d(kernel_size=(2, 4), stride=(2, 4), padding=0, dilation=1, ceil_mode=False)
  )
  (res_stack): Sequential(
    (0): ResidualBlock(
      (block): Sequential(
        (0): ConvBNAct(
          (conv): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affi

# Feature Extractor from Trained Model

In [ ]:
import torch
import torch.nn as nn

class ResidualEEGCNNFeatureExtractor(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.stem = model.stem
        self.res_stack = model.res_stack

        # remove final Linear(256 -> 2)
        self.feature_head = nn.Sequential(*list(model.head.children())[:-1])

    def forward(self, x):
        x = self.stem(x)
        x = self.res_stack(x)
        x = self.feature_head(x)
        return x

In [8]:
feature_extractor = ResidualEEGCNNFeatureExtractor(model).eval()

x = torch.randn(8, 1, 61, 1250)
with torch.no_grad():
    z = feature_extractor(x)

print(z.shape)

torch.Size([8, 256])


# Loading Data

In [9]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path("/content/drive/MyDrive/CSCI 5527 Project/Dataset Preprocessing/processed_eeg_dataset/processed_eeg_dataset")

X_eeg = np.load(DATA_DIR / "X_eeg.npy", mmap_mode="r")

metadata_sss = pd.read_csv(DATA_DIR / "eeg_metadata_with_sss.csv")
metadata_kss = pd.read_csv(DATA_DIR / "eeg_metadata_with_kss.csv")

y_phase1 = np.load(DATA_DIR / "y_labels.npy", allow_pickle=True)
groups = np.load(DATA_DIR / "groups.npy", allow_pickle=True)

print("X_eeg shape      :", X_eeg.shape)
print("metadata_sss     :", metadata_sss.shape)
print("metadata_kss     :", metadata_kss.shape)
print("phase1 labels    :", y_phase1.shape)
print("groups           :", groups.shape)

X_eeg shape      : (8300, 61, 1250)
metadata_sss     : (8300, 14)
metadata_kss     : (8300, 14)
phase1 labels    : (8300,)
groups           : (8300,)


In [10]:
from sklearn.model_selection import GroupShuffleSplit

SEED = 42
TRAIN_SIZE = 0.70
VAL_SIZE = 0.15
TEST_SIZE = 0.15

all_indices = np.arange(len(y_phase1))

gss_1 = GroupShuffleSplit(n_splits=1, train_size=TRAIN_SIZE, random_state=SEED)
train_idx, temp_idx = next(gss_1.split(all_indices, y_phase1, groups))

temp_y = y_phase1[temp_idx]
temp_groups = groups[temp_idx]
relative_val = VAL_SIZE / (VAL_SIZE + TEST_SIZE)

gss_2 = GroupShuffleSplit(n_splits=1, train_size=relative_val, random_state=SEED + 1)
temp_val_idx, temp_test_idx = next(gss_2.split(np.arange(len(temp_idx)), temp_y, temp_groups))

val_idx = temp_idx[temp_val_idx]
test_idx = temp_idx[temp_test_idx]

print("train:", len(train_idx))
print("val  :", len(val_idx))
print("test :", len(test_idx))

train: 5738
val  : 1246
test : 1316


In [11]:
import torch
from torch.utils.data import Dataset, DataLoader

class EEGFeatureDataset(Dataset):
    def __init__(self, x_eeg, metadata_df, indices):
        self.x_eeg = x_eeg
        self.metadata_df = metadata_df
        self.indices = np.asarray(indices, dtype=np.int64)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        global_idx = int(self.indices[idx])

        x = np.array(self.x_eeg[global_idx], dtype=np.float32, copy=True)
        x = torch.from_numpy(x).unsqueeze(0)   # (1, 61, 1250)

        meta = self.metadata_df.iloc[global_idx].to_dict()
        meta["global_index"] = global_idx

        return x, meta

In [12]:
metadata_target = metadata_kss.copy()
target_name = "KSS"

In [13]:
BATCH_SIZE = 64

train_dataset = EEGFeatureDataset(X_eeg, metadata_target, train_idx)
val_dataset = EEGFeatureDataset(X_eeg, metadata_target, val_idx)
test_dataset = EEGFeatureDataset(X_eeg, metadata_target, test_idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

In [14]:
x_batch, meta_batch = next(iter(train_loader))

print("x batch shape:", x_batch.shape)
print("meta keys:", meta_batch.keys())
print("first subject:", meta_batch["subject_id"][0])
print("first session:", meta_batch["session"][0])
print("first epoch:", meta_batch["epoch_index"][0])
print("first target:", meta_batch[target_name][0])

x batch shape: torch.Size([64, 1, 61, 1250])
meta keys: dict_keys(['subject_id', 'session', 'task', 'label', 'label_name', 'epoch_index', 'sfreq', 'n_channels', 'n_times', 'img_channels', 'img_height', 'img_width', 'file_path', 'KSS', 'global_index'])
first subject: sub-02
first session: ses-1
first epoch: tensor(0)
first target: tensor(nan, dtype=torch.float64)


In [15]:
from tqdm.auto import tqdm

def extract_epoch_features(loader, feature_extractor, device, target_name):
    rows = []

    feature_extractor.eval()
    with torch.no_grad():
        for x, meta in tqdm(loader):
            x = x.to(device)
            feats = feature_extractor(x).detach().cpu().numpy()

            batch_size = feats.shape[0]

            for i in range(batch_size):
                row = {
                    "global_index": int(meta["global_index"][i]),
                    "subject_id": meta["subject_id"][i],
                    "session": meta["session"][i],
                    "label_name": meta["label_name"][i],
                    "file_path": meta["file_path"][i],
                    "epoch_index": int(meta["epoch_index"][i]),
                    target_name: float(meta[target_name][i]) if pd.notna(meta[target_name][i]) else np.nan,
                }

                for j in range(feats.shape[1]):
                    row[f"f_{j}"] = float(feats[i, j])

                rows.append(row)

    return pd.DataFrame(rows)

In [16]:

feature_extractor = feature_extractor.to(DEVICE).eval()

train_feat_df = extract_epoch_features(train_loader, feature_extractor, DEVICE, target_name)
val_feat_df = extract_epoch_features(val_loader, feature_extractor, DEVICE, target_name)
test_feat_df = extract_epoch_features(test_loader, feature_extractor, DEVICE, target_name)

print(train_feat_df.shape)
train_feat_df.head()

  0%|          | 0/90 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/21 [00:00<?, ?it/s]

(5738, 263)


,global_index,subject_id,session,label_name,file_path,epoch_index,KSS,f_0,f_1,f_2,...,f_246,f_247,f_248,f_249,f_250,f_251,f_252,f_253,f_254,f_255
0,60,sub-02,ses-1,NS,data\sub-02\ses-1\eeg\sub-02_ses-1_task-eyesop...,0,NaN,0.048420,1.775064,-0.219845,...,-0.075633,0.148877,-0.058331,0.375994,0.369301,0.000806,-0.225646,0.036990,-0.236286,-0.216052
1,61,sub-02,ses-1,NS,data\sub-02\ses-1\eeg\sub-02_ses-1_task-eyesop...,1,NaN,0.030108,1.639028,-0.216687,...,-0.111317,0.207680,-0.065968,0.258174,0.362637,-0.040872,-0.212862,0.032450,-0.233867,-0.222573
2,62,sub-02,ses-1,NS,data\sub-02\ses-1\eeg\sub-02_ses-1_task-eyesop...,2,NaN,0.148415,0.940895,-0.238733,...,-0.090538,0.064769,-0.120224,0.111136,0.350660,-0.004258,-0.169977,0.247853,-0.276848,-0.191143
3,63,sub-02,ses-1,NS,data\sub-02\ses-1\eeg\sub-02_ses-1_task-eyesop...,3,NaN,-0.045613,2.046052,-0.237916,...,0.028021,0.043932,0.154100,0.501832,0.421358,0.034624,-0.256773,-0.066442,-0.201905,-0.230921
4,64,sub-02,ses-1,NS,data\sub-02\ses-1\eeg\sub-02_ses-1_task-eyesop...,4,NaN,0.103671,1.665081,-0.213632,...,-0.034245,0.099708,-0.111697,0.309238,0.479366,-0.030885,-0.219311,0.101341,-0.257783,-0.218550


In [17]:
train_feat_df = train_feat_df.dropna(subset=[target_name]).copy()
val_feat_df = val_feat_df.dropna(subset=[target_name]).copy()
test_feat_df = test_feat_df.dropna(subset=[target_name]).copy()

print(train_feat_df.shape, val_feat_df.shape, test_feat_df.shape)

(2918, 263) (706, 263) (236, 263)


In [18]:
OUT_DIR = DATA_DIR / "phase2_features"
OUT_DIR.mkdir(exist_ok=True)

train_feat_df.to_csv(OUT_DIR / f"train_epoch_features_residual_{target_name.lower()}.csv", index=False)
val_feat_df.to_csv(OUT_DIR / f"val_epoch_features_residual_{target_name.lower()}.csv", index=False)
test_feat_df.to_csv(OUT_DIR / f"test_epoch_features_residual_{target_name.lower()}.csv", index=False)

# Grouping Epochs into sessions

In [19]:
feature_cols = [c for c in train_feat_df.columns if c.startswith("f_")]
group_cols = ["subject_id", "session", "file_path"]

def build_session_sequences(feat_df, target_name):
    rows = []

    grouped = feat_df.sort_values(group_cols + ["epoch_index"]).groupby(group_cols)

    for (subject_id, session, file_path), g in grouped:
        g = g.sort_values("epoch_index")

        seq = g[feature_cols].to_numpy(dtype=np.float32)   # (T, D)
        target = float(g[target_name].iloc[0])

        rows.append({
            "subject_id": subject_id,
            "session": session,
            "file_path": file_path,
            "target": target,
            "num_epochs": len(g),
            "features": seq,
        })

    return pd.DataFrame(rows)

In [20]:
train_session_df = build_session_sequences(train_feat_df, target_name)
val_session_df = build_session_sequences(val_feat_df, target_name)
test_session_df = build_session_sequences(test_feat_df, target_name)

print(train_session_df.shape, val_session_df.shape, test_session_df.shape)
train_session_df.head()

(49, 6) (12, 6) (4, 6)


,subject_id,session,file_path,target,num_epochs,features
0,sub-39,ses-1,data\sub-39\ses-1\eeg\sub-39_ses-1_task-eyesop...,6.0,60,"[[0.12678544, 0.96558243, -0.24058695, 0.37508..."
1,sub-39,ses-2,data\sub-39\ses-2\eeg\sub-39_ses-2_task-eyesop...,3.0,60,"[[-0.27756512, 1.0233225, -0.27080694, -0.2671..."
2,sub-40,ses-1,data\sub-40\ses-1\eeg\sub-40_ses-1_task-eyesop...,5.0,60,"[[0.22106634, 0.93160033, -0.2348897, 0.416957..."
3,sub-40,ses-2,data\sub-40\ses-2\eeg\sub-40_ses-2_task-eyesop...,7.0,60,"[[-0.12446501, 0.5933352, -0.27840477, -0.0057..."
4,sub-41,ses-1,data\sub-41\ses-1\eeg\sub-41_ses-1_task-eyesop...,4.0,60,"[[0.031719092, 1.8091446, -0.22687827, -0.0314..."


In [21]:
import pickle

with open(OUT_DIR / f"train_session_features_residual_{target_name.lower()}.pkl", "wb") as f:
    pickle.dump(train_session_df, f)

with open(OUT_DIR / f"val_session_features_residual_{target_name.lower()}.pkl", "wb") as f:
    pickle.dump(val_session_df, f)

with open(OUT_DIR / f"test_session_features_residual_{target_name.lower()}.pkl", "wb") as f:
    pickle.dump(test_session_df, f)

In [22]:
def mean_pool_features(session_df):
    X = np.stack([row.mean(axis=0) for row in session_df["features"].values]).astype(np.float32)
    y = session_df["target"].to_numpy(dtype=np.float32)
    return X, y

X_train, y_train = mean_pool_features(train_session_df)
X_val, y_val = mean_pool_features(val_session_df)
X_test, y_test = mean_pool_features(test_session_df)

print(X_train.shape, y_train.shape)

(49, 256) (49,)


# Baseline

In [23]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr, spearmanr
import numpy as np

reg = Ridge(alpha=1.0)
reg.fit(X_train, y_train)

val_pred = reg.predict(X_val)
test_pred = reg.predict(X_test)

def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "Pearson": pearsonr(y_true, y_pred)[0] if len(np.unique(y_true)) > 1 else np.nan,
        "Spearman": spearmanr(y_true, y_pred)[0] if len(np.unique(y_true)) > 1 else np.nan,
    }

print("Validation:", regression_metrics(y_val, val_pred))
print("Test      :", regression_metrics(y_test, test_pred))

Validation: {'MAE': 1.1970371007919312, 'RMSE': np.float64(1.9038559310205574), 'Pearson': np.float32(0.17332312), 'Spearman': np.float64(0.33371190623595737)}
Test      : {'MAE': 1.4031001329421997, 'RMSE': np.float64(1.5473564246505156), 'Pearson': np.float32(0.75714004), 'Spearman': np.float64(0.7378647873726218)}


# MLP

In [24]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr, spearmanr

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class PooledFeatureDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_ds = PooledFeatureDataset(X_train, y_train)
val_ds = PooledFeatureDataset(X_val, y_val)
test_ds = PooledFeatureDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

In [25]:
class MLPRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

In [26]:
def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "Pearson": pearsonr(y_true, y_pred)[0] if len(np.unique(y_true)) > 1 else np.nan,
        "Spearman": spearmanr(y_true, y_pred)[0] if len(np.unique(y_true)) > 1 else np.nan,
    }

In [27]:
def evaluate_regression_model(model, loader, device):
    model.eval()
    ys, preds = [], []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            out = model(x).cpu().numpy()
            preds.extend(out.tolist())
            ys.extend(y.numpy().tolist())

    ys = np.array(ys, dtype=np.float32)
    preds = np.array(preds, dtype=np.float32)
    return ys, preds, regression_metrics(ys, preds)

In [28]:
input_dim = X_train.shape[1]
model = MLPRegressor(input_dim=input_dim, hidden_dim=128, dropout=0.3).to(DEVICE)

criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

best_val_rmse = float("inf")
best_state = None
patience = 20
stale = 0
num_epochs = 150

history = []

for epoch in range(1, num_epochs + 1):
    model.train()
    running_loss = 0.0

    for x, y in train_loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()
        pred = model(x)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x.size(0)

    train_loss = running_loss / len(train_loader.dataset)

    _, _, val_metrics = evaluate_regression_model(model, val_loader, DEVICE)
    val_rmse = val_metrics["RMSE"]

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        **{f"val_{k}": v for k, v in val_metrics.items()}
    })

    print(
        f"Epoch {epoch:03d} | "
        f"train_loss={train_loss:.4f} | "
        f"val_MAE={val_metrics['MAE']:.4f} | "
        f"val_RMSE={val_metrics['RMSE']:.4f} | "
        f"val_Pearson={val_metrics['Pearson']:.4f} | "
        f"val_Spearman={val_metrics['Spearman']:.4f}"
    )

    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        stale = 0
    else:
        stale += 1
        if stale >= patience:
            print("Early stopping triggered.")
            break

Epoch 001 | train_loss=35.7157 | val_MAE=5.6858 | val_RMSE=5.9927 | val_Pearson=-0.2331 | val_Spearman=-0.0183
Epoch 002 | train_loss=32.3515 | val_MAE=5.3302 | val_RMSE=5.6580 | val_Pearson=-0.4342 | val_Spearman=-0.2054
Epoch 003 | train_loss=28.2322 | val_MAE=4.8217 | val_RMSE=5.1855 | val_Pearson=-0.4671 | val_Spearman=-0.4034
Epoch 004 | train_loss=23.8511 | val_MAE=4.0974 | val_RMSE=4.5243 | val_Pearson=-0.4910 | val_Spearman=-0.4511
Epoch 005 | train_loss=16.2368 | val_MAE=3.4112 | val_RMSE=3.6632 | val_Pearson=-0.5060 | val_Spearman=-0.5244
Epoch 006 | train_loss=10.6059 | val_MAE=2.6536 | val_RMSE=2.7631 | val_Pearson=-0.5145 | val_Spearman=-0.4877
Epoch 007 | train_loss=7.7171 | val_MAE=1.8073 | val_RMSE=2.0790 | val_Pearson=-0.5085 | val_Spearman=-0.4877
Epoch 008 | train_loss=4.4514 | val_MAE=1.3864 | val_RMSE=2.0377 | val_Pearson=-0.4714 | val_Spearman=-0.4511
Epoch 009 | train_loss=5.2631 | val_MAE=1.4331 | val_RMSE=2.0923 | val_Pearson=-0.4161 | val_Spearman=-0.3080
Epoc

In [29]:
model.load_state_dict(best_state)

y_val_true, y_val_pred, val_metrics = evaluate_regression_model(model, val_loader, DEVICE)
y_test_true, y_test_pred, test_metrics = evaluate_regression_model(model, test_loader, DEVICE)

print("\nBest Validation Metrics:", val_metrics)
print("Test Metrics:", test_metrics)


Best Validation Metrics: {'MAE': 1.3863887786865234, 'RMSE': np.float64(2.037678215798293), 'Pearson': np.float32(-0.47142535), 'Spearman': np.float64(-0.451061147989261)}
Test Metrics: {'MAE': 1.6546281576156616, 'RMSE': np.float64(1.7696707630528006), 'Pearson': np.float32(0.69177675), 'Spearman': np.float64(0.7378647873726218)}


# GRU

In [30]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class SessionSequenceDataset(Dataset):
    def __init__(self, session_df):
        self.session_df = session_df.reset_index(drop=True)

    def __len__(self):
        return len(self.session_df)

    def __getitem__(self, idx):
        row = self.session_df.iloc[idx]
        x = torch.tensor(row["features"], dtype=torch.float32)   # (T, D)
        y = torch.tensor(row["target"], dtype=torch.float32)
        length = x.shape[0]
        return x, y, length

In [31]:
from torch.nn.utils.rnn import pad_sequence

def collate_session_sequences(batch):
    xs, ys, lengths = zip(*batch)

    lengths = torch.tensor(lengths, dtype=torch.long)
    ys = torch.tensor(ys, dtype=torch.float32)

    xs_padded = pad_sequence(xs, batch_first=True)   # (B, T_max, D)

    return xs_padded, ys, lengths

In [32]:
train_seq_ds = SessionSequenceDataset(train_session_df)
val_seq_ds = SessionSequenceDataset(val_session_df)
test_seq_ds = SessionSequenceDataset(test_session_df)

train_seq_loader = DataLoader(
    train_seq_ds,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_session_sequences
)

val_seq_loader = DataLoader(
    val_seq_ds,
    batch_size=16,
    shuffle=False,
    collate_fn=collate_session_sequences
)

test_seq_loader = DataLoader(
    test_seq_ds,
    batch_size=16,
    shuffle=False,
    collate_fn=collate_session_sequences
)

In [33]:
import torch.nn as nn

class GRURegressor(nn.Module):
    def __init__(self, input_dim=256, hidden_dim=128, num_layers=1, dropout=0.2, bidirectional=False):
        super().__init__()

        gru_dropout = dropout if num_layers > 1 else 0.0

        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=gru_dropout,
            bidirectional=bidirectional
        )

        out_dim = hidden_dim * (2 if bidirectional else 1)

        self.head = nn.Sequential(
            nn.Linear(out_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, h_n = self.gru(packed)

        if self.gru.bidirectional:
            final_hidden = torch.cat([h_n[-2], h_n[-1]], dim=1)
        else:
            final_hidden = h_n[-1]

        out = self.head(final_hidden).squeeze(-1)
        return out

In [34]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr, spearmanr

def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "Pearson": pearsonr(y_true, y_pred)[0] if len(np.unique(y_true)) > 1 else np.nan,
        "Spearman": spearmanr(y_true, y_pred)[0] if len(np.unique(y_true)) > 1 else np.nan,
    }

In [35]:
def evaluate_sequence_model(model, loader, device):
    model.eval()
    ys, preds = [], []

    with torch.no_grad():
        for x, y, lengths in loader:
            x = x.to(device)
            y = y.to(device)
            lengths = lengths.to(device)

            out = model(x, lengths)

            preds.extend(out.detach().cpu().numpy().tolist())
            ys.extend(y.detach().cpu().numpy().tolist())

    ys = np.array(ys, dtype=np.float32)
    preds = np.array(preds, dtype=np.float32)
    return ys, preds, regression_metrics(ys, preds)

In [36]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

gru_model = GRURegressor(
    input_dim=256,
    hidden_dim=128,
    num_layers=1,
    dropout=0.3,
    bidirectional=False
).to(DEVICE)

criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(gru_model.parameters(), lr=1e-3, weight_decay=1e-4)

best_val_rmse = float("inf")
best_state = None
patience = 20
stale = 0
num_epochs = 150

gru_history = []

for epoch in range(1, num_epochs + 1):
    gru_model.train()
    running_loss = 0.0

    for x, y, lengths in train_seq_loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        lengths = lengths.to(DEVICE)

        optimizer.zero_grad()
        pred = gru_model(x, lengths)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x.size(0)

    train_loss = running_loss / len(train_seq_loader.dataset)

    _, _, val_metrics = evaluate_sequence_model(gru_model, val_seq_loader, DEVICE)
    val_rmse = val_metrics["RMSE"]

    gru_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        **{f"val_{k}": v for k, v in val_metrics.items()}
    })

    print(
        f"Epoch {epoch:03d} | "
        f"train_loss={train_loss:.4f} | "
        f"val_MAE={val_metrics['MAE']:.4f} | "
        f"val_RMSE={val_metrics['RMSE']:.4f} | "
        f"val_Pearson={val_metrics['Pearson']:.4f} | "
        f"val_Spearman={val_metrics['Spearman']:.4f}"
    )

    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        best_state = {k: v.cpu().clone() for k, v in gru_model.state_dict().items()}
        stale = 0
    else:
        stale += 1
        if stale >= patience:
            print("Early stopping triggered.")
            break

Epoch 001 | train_loss=25.6191 | val_MAE=3.3913 | val_RMSE=3.6548 | val_Pearson=-0.3650 | val_Spearman=-0.4731
Epoch 002 | train_loss=7.8415 | val_MAE=1.4198 | val_RMSE=1.9083 | val_Pearson=-0.3040 | val_Spearman=-0.3484
Epoch 003 | train_loss=4.1638 | val_MAE=1.3360 | val_RMSE=1.9699 | val_Pearson=-0.2845 | val_Spearman=-0.2970
Epoch 004 | train_loss=4.3527 | val_MAE=1.3337 | val_RMSE=1.9008 | val_Pearson=-0.2535 | val_Spearman=-0.3117
Epoch 005 | train_loss=4.4942 | val_MAE=1.9061 | val_RMSE=2.0993 | val_Pearson=-0.2393 | val_Spearman=-0.2164
Epoch 006 | train_loss=4.4548 | val_MAE=1.9200 | val_RMSE=2.1107 | val_Pearson=-0.2469 | val_Spearman=-0.1870
Epoch 007 | train_loss=4.6958 | val_MAE=1.5458 | val_RMSE=1.9417 | val_Pearson=-0.2594 | val_Spearman=-0.1650
Epoch 008 | train_loss=4.4179 | val_MAE=1.3613 | val_RMSE=1.9199 | val_Pearson=-0.2644 | val_Spearman=-0.1650
Epoch 009 | train_loss=3.9395 | val_MAE=1.4898 | val_RMSE=1.9414 | val_Pearson=-0.2639 | val_Spearman=-0.1650
Epoch 010

In [37]:
gru_model.load_state_dict(best_state)

y_val_true, y_val_pred, val_metrics = evaluate_sequence_model(gru_model, val_seq_loader, DEVICE)
y_test_true, y_test_pred, test_metrics = evaluate_sequence_model(gru_model, test_seq_loader, DEVICE)

print("\nBest Validation Metrics:", val_metrics)
print("Test Metrics:", test_metrics)


Best Validation Metrics: {'MAE': 1.3337278366088867, 'RMSE': np.float64(1.9007512188603604), 'Pearson': np.float32(-0.25353596), 'Spearman': np.float64(-0.3117089234072129)}
Test Metrics: {'MAE': 1.7348777055740356, 'RMSE': np.float64(1.7731587867354957), 'Pearson': np.float32(0.62201846), 'Spearman': np.float64(0.632455532033676)}


# LSTM

In [39]:
import torch.nn as nn

class LSTMRegressor(nn.Module):
    def __init__(self, input_dim=256, hidden_dim=128, num_layers=1, dropout=0.2, bidirectional=False):
        super().__init__()

        lstm_dropout = dropout if num_layers > 1 else 0.0

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=lstm_dropout,
            bidirectional=bidirectional
        )

        out_dim = hidden_dim * (2 if bidirectional else 1)

        self.head = nn.Sequential(
            nn.Linear(out_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, (h_n, _) = self.lstm(packed)

        if self.lstm.bidirectional:
            final_hidden = torch.cat([h_n[-2], h_n[-1]], dim=1)
        else:
            final_hidden = h_n[-1]

        out = self.head(final_hidden).squeeze(-1)
        return out

In [40]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

lstm_model = LSTMRegressor(
    input_dim=256,
    hidden_dim=128,
    num_layers=1,
    dropout=0.3,
    bidirectional=False
).to(DEVICE)

criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(lstm_model.parameters(), lr=1e-3, weight_decay=1e-4)

best_val_rmse = float("inf")
best_state = None
patience = 20
stale = 0
num_epochs = 150

lstm_history = []

for epoch in range(1, num_epochs + 1):
    lstm_model.train()
    running_loss = 0.0

    for x, y, lengths in train_seq_loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        lengths = lengths.to(DEVICE)

        optimizer.zero_grad()
        pred = lstm_model(x, lengths)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x.size(0)

    train_loss = running_loss / len(train_seq_loader.dataset)

    _, _, val_metrics = evaluate_sequence_model(lstm_model, val_seq_loader, DEVICE)
    val_rmse = val_metrics["RMSE"]

    lstm_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        **{f"val_{k}": v for k, v in val_metrics.items()}
    })

    print(
        f"Epoch {epoch:03d} | "
        f"train_loss={train_loss:.4f} | "
        f"val_MAE={val_metrics['MAE']:.4f} | "
        f"val_RMSE={val_metrics['RMSE']:.4f} | "
        f"val_Pearson={val_metrics['Pearson']:.4f} | "
        f"val_Spearman={val_metrics['Spearman']:.4f}"
    )

    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        best_state = {k: v.cpu().clone() for k, v in lstm_model.state_dict().items()}
        stale = 0
    else:
        stale += 1
        if stale >= patience:
            print("Early stopping triggered.")
            break

Epoch 001 | train_loss=31.6868 | val_MAE=4.6618 | val_RMSE=5.0329 | val_Pearson=-0.2209 | val_Spearman=-0.0770
Epoch 002 | train_loss=17.3341 | val_MAE=2.7935 | val_RMSE=2.9246 | val_Pearson=-0.4504 | val_Spearman=-0.5244
Epoch 003 | train_loss=4.6956 | val_MAE=1.3209 | val_RMSE=1.9487 | val_Pearson=-0.3419 | val_Spearman=-0.4291
Epoch 004 | train_loss=5.4071 | val_MAE=1.3630 | val_RMSE=2.0056 | val_Pearson=-0.2958 | val_Spearman=-0.2824
Epoch 005 | train_loss=4.0515 | val_MAE=1.7158 | val_RMSE=2.0030 | val_Pearson=-0.2764 | val_Spearman=-0.2824
Epoch 006 | train_loss=4.4516 | val_MAE=2.0473 | val_RMSE=2.2026 | val_Pearson=-0.2716 | val_Spearman=-0.2347
Epoch 007 | train_loss=4.8479 | val_MAE=1.8955 | val_RMSE=2.1001 | val_Pearson=-0.2735 | val_Spearman=-0.2640
Epoch 008 | train_loss=4.2895 | val_MAE=1.3497 | val_RMSE=1.9197 | val_Pearson=-0.2764 | val_Spearman=-0.2640
Epoch 009 | train_loss=4.5904 | val_MAE=1.3215 | val_RMSE=1.9353 | val_Pearson=-0.2781 | val_Spearman=-0.2640
Epoch 01

In [41]:
lstm_model.load_state_dict(best_state)

y_val_true, y_val_pred, val_metrics = evaluate_sequence_model(lstm_model, val_seq_loader, DEVICE)
y_test_true, y_test_pred, test_metrics = evaluate_sequence_model(lstm_model, test_seq_loader, DEVICE)

print("\nBest Validation Metrics:", val_metrics)
print("Test Metrics:", test_metrics)


Best Validation Metrics: {'MAE': 1.3497428894042969, 'RMSE': np.float64(1.9196798922999525), 'Pearson': np.float32(-0.27643406), 'Spearman': np.float64(-0.2640357939449333)}
Test Metrics: {'MAE': 1.7393532991409302, 'RMSE': np.float64(1.7786523717384932), 'Pearson': np.float32(0.49690565), 'Spearman': np.float64(0.21081851067789195)}
